# Microproyecto 2: Generación de Código con Transformers

**Curso:** Modelos Avanzados para el Procesamiento de Lenguaje Natural
**Equipo 4**
**Integrantes:** Juan Sebastián Páez Cortés, Julian David Parada Galvis, Luis Carlos Prada Socha, Jean Pierre Betancourt Padilla
**Fecha:** octubre de 2026

---

## 1. Portada e introducción

### Resumen

En este microproyecto entrenamos modelos tipo GPT (Transformer decoder-only) para generar
código en Python. El modelo aprende a predecir el siguiente token de un archivo de código y,
con eso, puede continuar un fragmento que le demos como entrada.

Comparamos cinco configuraciones para ver cuánto depende el resultado del tamaño del modelo
y de si se entrena desde cero o se parte de pesos ya preentrenados:

| # | Configuración | Modelo | Entrenamiento |
| :-: | :--- | :--- | :--- |
| 1 | Desde cero | GPT decoder-only en PyTorch | 3 épocas desde pesos aleatorios |
| 2 | Baseline 135M | SmolLM2-135M | Sin entrenar |
| 3 | Finetuning 135M | SmolLM2-135M | Full finetuning, 1 época |
| 4 | Baseline 360M | SmolLM2-360M | Sin entrenar |
| 5 | Finetuning 360M | SmolLM2-360M | Full finetuning, 1 época |

### Datos

Usamos `codeparrot/codeparrot-clean` en modo streaming. Nos quedamos solo con archivos de
entre 1.000 y 2.000 caracteres, tomamos 8.000 ejemplos y los dividimos en 6.400 de
entrenamiento, 800 de validación y 800 de prueba como lo pide el proyecto. Todos los modelos
usan el tokenizador de SmolLM2 (49.152 tokens) y un contexto de 256 tokens. El filtro, el split
y el contexto son los que exige la guía del proyecto.

### Evaluación

- Perplejidad sobre los 800 ejemplos de prueba.
- Porcentaje de código sintácticamente válido con `ast` en 200 ejemplos.
- Tres ejemplos generados por modelo con nuestro propio bucle autorregresivo.

### Entorno

El notebook corre completo en Google Colab con GPU L4. Para el modelo de 360M usamos
precisión mixta, acumulación de gradientes y gradient checkpointing, que es la estrategia de
memoria que recomienda el personal del curso. Entre un modelo y el siguiente liberamos la
memoria de la GPU.

Todo el pipeline es reproducible: fijamos la semilla y la revisión del dataset y del
tokenizador, de modo que dos ejecuciones producen los mismos 8.000 archivos y el mismo split.


## 2. Librerías

Dejamos listo el entorno de ejecución antes de tocar los datos, siguiendo lo que pide la guía
del proyecto.


Instalamos las versiones de las librerías de Hugging Face que se usan en el proyecto.


In [ ]:
!pip install -q "transformers>=4.44,<5" "datasets>=2.20,<4" "accelerate>=0.33"


Importamos lo necesario y comprobamos las versiones cargadas; `ast` lo usaremos para la
validación sintáctica que pide la guía.


In [ ]:
import ast
import gc
import math
import os
import random
import time
import warnings

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # workspace fijo para que las multiplicaciones de matrices den siempre lo mismo

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import datasets
import transformers
from datasets import load_dataset
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # evita problemas con los avisos de fork en Colab

print("Entorno cargado")
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("numpy:", np.__version__)


Configuramos la GPU, la precisión mixta, las constantes que fija el enunciado (tokenizador
SmolLM2, contexto 256 y batch efectivo 16) y la carpeta de checkpoints.


In [ ]:
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()

if USE_BF16:  # elige precision mixta segun la GPU de Colab (T4 o L4)
    AMP_DTYPE = torch.bfloat16
    NOMBRE_AMP = "bfloat16"
    USE_SCALER = False
else:
    AMP_DTYPE = torch.float16
    NOMBRE_AMP = "float16"
    USE_SCALER = True

if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM:", round(props.total_memory / 1024**3, 1), "GB")
else:
    print("No se detecto GPU, el entrenamiento sera con CPU y puede ser muy lento")

print("AMP:", NOMBRE_AMP)
print("GradScaler:", USE_SCALER)

SMOL_135M = "HuggingFaceTB/SmolLM2-135M"
SMOL_360M = "HuggingFaceTB/SmolLM2-360M"
TOKENIZER_ID = SMOL_135M  # uno solo para las cinco configuraciones porque la PPL se mide por token

DATASET_ID = "codeparrot/codeparrot-clean"
DATASET_REVISION = "35a59fb025bc0a102f7d96eac09d145b896d487b"
SMOL_135M_REVISION = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"
SMOL_360M_REVISION = "f8027fd0eaeea54caa13c31d31b9fdc459c38b49"

CTX = 256
VOCAB_SIZE = 49152
EOS_TOKEN = "<|endoftext|>"

MIN_CHARS = 1000
MAX_CHARS = 2000
N_TOTAL = 8000
N_TRAIN = 6400
N_VAL = 800
N_TEST = 800

BATCH_EFECTIVO = 16
CLIP_GRAD_NORM = 1.0

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/microproyecto_2"
os.makedirs(DRIVE_DIR, exist_ok=True)

print("Checkpoints:", DRIVE_DIR)

if "HF_TOKEN" in os.environ:
    print("HF_TOKEN, configurado")
else:
    print("HF_TOKEN: no esta en las variables de entorno")


Fijamos la semilla para que los resultados sean reproducibles y creamos la función que libera
la GPU entre modelos, como indica el documento de checkpoints.


In [ ]:
def fijar_semilla(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def liberar_vram_total(modelo=None, optimizador=None):
    if modelo is not None:
        del modelo
    if optimizador is not None:
        del optimizador
    gc.collect()
    torch.cuda.empty_cache()


fijar_semilla()
print("Semilla:", SEED)


## 3. Carga del dataset y exploración

Trabajamos con `codeparrot/codeparrot-clean`, el dataset que indica el proyecto. Lo cargamos en
modo streaming para no descargar los 5 GB completos: los ejemplos se van pidiendo a medida que
los necesitamos.

Primero miramos la estructura del dataset, después qué tan largos son los archivos (el
enunciado solo acepta los que tienen entre 1.000 y 2.000 caracteres) y, por último, revisamos
licencias, repos y duplicados para saber con qué datos estamos trabajando.


Cargamos el dataset en streaming, fijando la revisión para que el contenido no cambie entre
ejecuciones, y miramos qué columnas trae.


In [ ]:
dataset = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True,
    revision=DATASET_REVISION,
)
ejemplo = next(iter(dataset))

print(dataset)
print("Columnas:", list(ejemplo.keys()))
print("Revision:", DATASET_REVISION)


Mostramos un archivo de ejemplo para ver cómo viene el campo `content`.


In [ ]:
print("repo:", ejemplo["repo_name"])
print("archivo:", ejemplo["path"])
print("caracteres:", len(ejemplo["content"]))
print("lineas:", ejemplo["content"].count("\n"))
print()
print(ejemplo["content"][:600])


Medimos 2.000 archivos y guardamos la muestra completa para analizarla sin volver al stream.


In [ ]:
N_MUESTRA = 2000
muestra = []

inicio = time.time()
for i, ejemplo in enumerate(dataset):
    if i == N_MUESTRA:
        break
    muestra.append(ejemplo)

longitudes = []
for e in muestra:
    longitudes.append(len(e["content"]))

print("Archivos medidos:", len(muestra))


Contamos cuántos archivos caen en el rango que pide el enunciado y resumimos la distribución.


In [ ]:
dentro_rango = 0
for largo in longitudes:
    if largo >= MIN_CHARS and largo <= MAX_CHARS:
        dentro_rango += 1

ordenadas = sorted(longitudes)
n = len(ordenadas)

print("En el rango:", dentro_rango, "de", n)
print("Porcentaje:", round(100 * dentro_rango / n, 1), "%")
print("Minimo:", ordenadas[0])
print("P25:", ordenadas[n // 4])
print("Mediana:", ordenadas[n // 2])
print("P75:", ordenadas[3 * n // 4])
print("P90:", ordenadas[9 * n // 10])
print("P99:", ordenadas[99 * n // 100])
print("Maximo:", ordenadas[-1])
print("Media:", round(sum(longitudes) / n, 1))


Graficamos la distribución completa de longitudes.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(longitudes, bins=60)
plt.axvline(MIN_CHARS, color="red", linestyle="--")
plt.axvline(MAX_CHARS, color="red", linestyle="--")
plt.title("Longitud de los archivos de codeparrot-clean")
plt.xlabel("caracteres")
plt.ylabel("cantidad de archivos")
plt.show()


La gráfica anterior muestra la distribución completa, pero la cola de archivos muy largos
aplasta la zona que nos interesa. Por eso repetimos el histograma recortado a 4.000 caracteres:
no estamos cambiando los datos, solo el zoom de la figura.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(longitudes, bins=60, range=(0, 4000))
plt.axvline(MIN_CHARS, color="red", linestyle="--")
plt.axvline(MAX_CHARS, color="red", linestyle="--")
plt.title("Longitud de los archivos (vista hasta 4.000 caracteres)")
plt.xlabel("caracteres")
plt.ylabel("cantidad de archivos")
plt.xlim(0, 4000)
plt.show()


Revisamos el tipo de dato y un valor de ejemplo de cada columna.


In [ ]:
print("Columna           Tipo       Ejemplo")
for nombre in muestra[0]:
    valor = muestra[0][nombre]
    texto = str(valor).replace("\n", " ")
    print(nombre.ljust(17), type(valor).__name__.ljust(10), texto[:45])


Revisamos si hay archivos vacíos o con contenido nulo.


In [ ]:
vacios = 0
solo_espacios = 0

for e in muestra:
    contenido = e["content"]
    if contenido is None or len(contenido) == 0:
        vacios += 1
    elif len(contenido.strip()) == 0:
        solo_espacios += 1

print("Vacios o nulos:", vacios)
print("Solo espacios:", solo_espacios)


Comparamos el archivo más corto de la muestra, uno que cae en el rango y uno largo. Ojo: casi
no aparecen archivos de menos de 1.000 caracteres, así que el límite que de verdad descarta es
el de 2.000.


In [ ]:
menores_1000 = 0
for e in muestra:
    if len(e["content"]) < MIN_CHARS:
        menores_1000 += 1

print("Archivos con menos de 1.000 caracteres:", menores_1000)

mas_corto = muestra[0]
en_rango = None
largo = None

for e in muestra:
    n = len(e["content"])
    if n < len(mas_corto["content"]):
        mas_corto = e
    if en_rango is None and n >= MIN_CHARS and n <= MAX_CHARS:
        en_rango = e
    if largo is None and n > MAX_CHARS:
        largo = e

casos = [("MAS CORTO", mas_corto), ("EN RANGO", en_rango), ("LARGO", largo)]

for nombre, e in casos:
    if e is None:
        print("=====", nombre, "- no se encontro en la muestra =====")
        print()
        continue
    print("=====", nombre, "-", len(e["content"]), "caracteres =====")
    print(e["content"][:300])
    print()


Contamos cuántas veces aparece cada construcción típica de Python.


In [ ]:
palabras = ["def ", "class ", "import ", "return ", "print("]
conteos = {}

for palabra in palabras:
    conteos[palabra] = 0

for e in muestra:
    for palabra in palabras:
        conteos[palabra] += e["content"].count(palabra)

valores = []
for palabra in palabras:
    print(palabra.strip(), conteos[palabra])
    valores.append(conteos[palabra])

plt.figure(figsize=(8, 3))
plt.barh(palabras, valores)
plt.title("Construcciones de Python en 2.000 archivos")
plt.xlabel("apariciones")
plt.show()


Revisamos cuáles son las licencias más comunes.


In [ ]:
licencias = {}

for e in muestra:
    lic = str(e["license"])
    if lic in licencias:
        licencias[lic] += 1
    else:
        licencias[lic] = 1

pares = []
for lic, cantidad in licencias.items():
    pares.append((cantidad, lic))
pares.sort(reverse=True)

nombres = []
valores = []
for cantidad, lic in pares[:5]:
    print(cantidad, lic)
    nombres.append(lic[:30])
    valores.append(cantidad)

plt.figure(figsize=(8, 3))
plt.barh(nombres, valores)
plt.title("Licencias mas comunes")
plt.xlabel("archivos")
plt.show()


Revisamos si los archivos vienen de pocos repos y cuántos están repetidos.


In [ ]:
repos = {}

for e in muestra:
    repo = e["repo_name"]
    if repo in repos:
        repos[repo] += 1
    else:
        repos[repo] = 1

pares = []
for repo, cantidad in repos.items():
    pares.append((cantidad, repo))
pares.sort(reverse=True)

print("Repos distintos:", len(repos))
for cantidad, repo in pares[:5]:
    print(cantidad, repo)

vistos = set()
repetidos = 0
for e in muestra:
    if e["hash"] in vistos:
        repetidos += 1
    else:
        vistos.add(e["hash"])

print("Archivos repetidos por hash:", repetidos)


Estimamos cuántos archivos hay que revisar para juntar los 8.000 que pide el enunciado.


In [ ]:
# si 399 de cada 2.000 sirven, para 8.000 necesitamos 8000 x 2000 / 399
acumulado = []
total = 0

for largo in longitudes:
    if largo >= MIN_CHARS and largo <= MAX_CHARS:
        total += 1
    acumulado.append(total)

esperado = round(N_TOTAL * len(longitudes) / dentro_rango)

plt.figure(figsize=(8, 4))
plt.plot(acumulado, label="muestra")
plt.plot([0, esperado], [0, N_TOTAL], linestyle="--", label="proyeccion")
plt.axhline(N_TOTAL, color="red", linestyle="--")
plt.xlim(0, esperado * 1.05)
plt.ylim(0, N_TOTAL * 1.1)
plt.title("Archivos utiles acumulados")
plt.xlabel("archivos revisados")
plt.ylabel("archivos en rango")
plt.legend()
plt.show()

print("Para 8.000 utiles hay que revisar cerca de", esperado, "archivos")


### Conclusiones de la exploración

**Solo 2 de cada 10 archivos sirven.** De los 2.000 que revisamos, 399 (19,9 %) caen entre
1.000 y 2.000 caracteres. Para juntar los 8.000 que pide el enunciado hay que recorrer unos
40.100 archivos, o sea cinco veces más de lo que costaría si todos calificaran. En tiempo no
duele: 2.000 archivos tardaron 1,4 s, así que el filtro completo toma unos 30 s.

**La distribución es de cola larga.** La media (10.263,6) es 2,2 veces la mediana (4.733). Si la
distribución fuera simétrica, media y mediana coincidirían; que estén tan separadas significa
que unos pocos archivos gigantes arrastran el promedio. El máximo es 456.237 caracteres: 96
veces la mediana y 445 veces el archivo más corto. Por eso la media no describe estos datos y
preferimos la mediana y los percentiles para hablar del "archivo típico".

**El límite inferior no filtra nada.** El archivo más corto de la muestra tiene 1.027 caracteres
y no apareció ni uno por debajo de 1.000: codeparrot-clean ya viene pre-filtrado. Todo el
descarte lo hace el techo de 2.000, que deja fuera a 8 de cada 10 archivos.

**El techo también sesga el contenido.** Además de limitar el tamaño, tiende a quedarse con
archivos de poca cabecera: los que traen bloques grandes de licencia o comentarios iniciales
suelen pasar los 2.000 caracteres y quedan fuera. Es una limitación del protocolo, no algo que
podamos corregir sin incumplirlo, pero conviene recordarla al interpretar los resultados.

**Qué clase de código esperar.** En 2.000 archivos contamos 27.325 `def`, 13.547 `import` y
17.149 `return`, contra solo 1.297 `print`. Domina la definición de funciones dentro de módulos,
no scripts que corren de principio a fin. Por eso esperamos que los modelos finetuneados generen
funciones y clases plausibles antes que programas completos, y que el modelo desde cero salga
mucho más plano y repetitivo.

**Calidad y procedencia.** No encontramos `content` vacíos, nulos ni duplicados por `hash`. Los
2.000 archivos vienen de 1.855 repos distintos (máximo 5 de un mismo repo), así que el split
aleatorio no debería mezclar el mismo proyecto entre train y test. Las licencias son en su
mayoría permisivas (apache 421, mit 366, bsd 347), con algo de copyleft (gpl-3 347, agpl-3 186);
como los bloques conservan las cabeceras, el modelo también aprende a emitirlas.

En volumen, los 8.000 archivos filtrados dieron 3.460.082 tokens, que se empacan en 13.514
bloques de 256. Eso es lo que alimenta el entrenamiento de la sección 6.


## 4. Preprocesamiento

Aplicamos el filtro de longitud que pide el enunciado, tokenizamos con el tokenizador de
SmolLM2 y empacamos los tokens en bloques fijos de 256, que es el contexto que van a usar los
cinco modelos.


Cargamos el tokenizador de SmolLM2 con su revisión fija y revisamos su vocabulario.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, revision=SMOL_135M_REVISION)

print("Tokenizador:", TOKENIZER_ID)
print("Vocabulario:", len(tokenizer))
print("Token de fin:", tokenizer.eos_token, "-", tokenizer.eos_token_id)
print("Revision:", SMOL_135M_REVISION)


Filtramos el stream y nos quedamos con los 8.000 primeros archivos del rango.


In [ ]:
textos = []

inicio = time.time()
for e in dataset:
    largo = len(e["content"])
    if largo >= MIN_CHARS and largo <= MAX_CHARS:
        textos.append(e["content"])
    if len(textos) == N_TOTAL:
        break

print("Archivos filtrados:", len(textos))
print("Tiempo:", round(time.time() - inicio, 1), "segundos")

if len(textos) == N_TOTAL:
    print("Se encontraron los", N_TOTAL, "archivos que pide el enunciado")
else:
    print("ADVERTENCIA: se esperaban", N_TOTAL, "archivos y se encontraron", len(textos))


Tokenizamos cada archivo y le agregamos el token de fin de texto.


In [ ]:
ids_por_texto = []
total_tokens = 0

for texto in textos:
    ids = tokenizer.encode(texto, add_special_tokens=False)
    ids.append(tokenizer.eos_token_id)
    ids_por_texto.append(ids)
    total_tokens += len(ids)

print("Archivos tokenizados:", len(ids_por_texto))
print("Tokens en total:", total_tokens)
print("Promedio por archivo:", round(total_tokens / len(ids_por_texto), 1))


Mostramos cómo se ve un archivo después de tokenizarlo.


In [ ]:
primeros_ids = ids_por_texto[0][:40]
print("Primeros tokens:", primeros_ids)
print("Como texto:", tokenizer.decode(primeros_ids))


Empacamos los tokens en bloques fijos de 256 y probamos la función con 50 archivos.


In [ ]:
def empacar(lista_ids, contexto):
    bloques = []
    buffer = []
    for ids in lista_ids:
        buffer.extend(ids)
        while len(buffer) >= contexto:
            bloques.append(buffer[:contexto])
            buffer = buffer[contexto:]
    return bloques


bloques_prueba = empacar(ids_por_texto[:50], CTX)

print("Bloques de prueba:", len(bloques_prueba))
print("Largo de cada bloque:", len(bloques_prueba[0]))
print("Bloques si se empacaran los 8.000 juntos:", total_tokens // CTX)
print()
print(tokenizer.decode(bloques_prueba[0][:80]))


## 5. Split train / validación / prueba

Partimos los 8.000 textos en 6.400 de entrenamiento, 800 de validación y 800 de prueba. Primero
mezclamos con la semilla fija para que el corte no dependa del orden en que llegan los archivos
del stream, y después empacamos cada parte por separado en bloques de 256 tokens.


Mezclamos los textos y sus tokens con la semilla fija.


In [ ]:
rng = random.Random(SEED)
orden = list(range(N_TOTAL))
rng.shuffle(orden)

textos_mezclados = []
ids_mezclados = []

for i in orden:
    textos_mezclados.append(textos[i])
    ids_mezclados.append(ids_por_texto[i])

print("Textos mezclados:", len(textos_mezclados))
print("Semilla:", SEED)


Partimos en 6.400 de entrenamiento, 800 de validación y 800 de prueba.


In [ ]:
textos_train = textos_mezclados[:N_TRAIN]
textos_val = textos_mezclados[N_TRAIN:N_TRAIN + N_VAL]
textos_test = textos_mezclados[N_TRAIN + N_VAL:]

ids_train = ids_mezclados[:N_TRAIN]
ids_val = ids_mezclados[N_TRAIN:N_TRAIN + N_VAL]
ids_test = ids_mezclados[N_TRAIN + N_VAL:]

print("Train:", len(textos_train))
print("Validacion:", len(textos_val))
print("Prueba:", len(textos_test))
print("Total:", len(textos_train) + len(textos_val) + len(textos_test))


Verificamos que ningún texto quede en dos partes a la vez.


In [ ]:
set_train = set(textos_train)
set_val = set(textos_val)
set_test = set(textos_test)

print("Repetidos train-validacion:", len(set_train & set_val))
print("Repetidos train-prueba:", len(set_train & set_test))
print("Repetidos validacion-prueba:", len(set_val & set_test))
print("Textos unicos:", len(set_train | set_val | set_test))


Empacamos cada parte por separado en bloques de 256 tokens.


In [ ]:
bloques_train = empacar(ids_train, CTX)
bloques_val = empacar(ids_val, CTX)
bloques_test = empacar(ids_test, CTX)

print("Bloques train:", len(bloques_train))
print("Bloques validacion:", len(bloques_val))
print("Bloques prueba:", len(bloques_test))
print("Bloques totales:", len(bloques_train) + len(bloques_val) + len(bloques_test))


Convertimos los bloques a tensores y revisamos sus formas.


In [ ]:
X_train = torch.tensor(bloques_train, dtype=torch.long)
X_val = torch.tensor(bloques_val, dtype=torch.long)
X_test = torch.tensor(bloques_test, dtype=torch.long)

print("Train:", X_train.shape, X_train.dtype)
print("Validacion:", X_val.shape, X_val.dtype)
print("Prueba:", X_test.shape, X_test.dtype)
print("Tokens de entrenamiento:", X_train.numel())


Revisamos cuántos tokens se pierden al final de cada parte por no completar un bloque.


In [ ]:
partes = [
    ("train", ids_train, bloques_train),
    ("validacion", ids_val, bloques_val),
    ("prueba", ids_test, bloques_test),
]

for nombre, ids, bloques in partes:
    total = 0
    for x in ids:
        total += len(x)
    usados = len(bloques) * CTX
    print(nombre, "- tokens:", total, "| usados:", usados, "| descartados:", total - usados)
